In [ ]:

!pip install --upgrade transformers
!pip install git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 92.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-odudwjjf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-odudwjjf
  Resolved https://github.com/huggingface/transformers.git to commit d3f05911abb216047a00aba79c8543c73633db05
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.10.0.dev0-py3-none-any.whl size=12161226 sha256=81772798ae9df645c6f4a73237cfee32f46e6c41d61fa73b1eefa36806d5b7e0
  Stored in directory: /tmp/pip-ephem-wheel-cache-jvabt9rf/wheels/54/cb/3f/83103de5575c534436d6a4686686dead45

In [ ]:
# 1. Установка системных пакетов и библиотек для ML-перевода
!pip install -q sacrebleu accelerate

# 2. Клонирование твоего репозитория (подставь свою ссылку)
# Если репозиторий приватный, используй personal access token: https://<token>@github.com/username/repo.git
!git clone https://github.com/weissv/task1 ./workspace

# 3. Переходим в рабочую директорию
%cd ./workspace/task2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
Cloning into './workspace'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 42 (delta 11), reused 39 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 111.07 KiB | 6.17 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/workspace


In [ ]:
!python download_weights.py

Fetching 9 files:   0% 0/9 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 9 files: 100% 9/9 [01:12<00:00,  8.04s/it]
Download complete: 100% 10.2G/10.2G [01:12<00:00, 200MB/s]                weights downloaded to: /content/workspace/weights
Download complete: 100% 10.2G/10.2G [01:12<00:00, 142MB/s]


In [ ]:
%%writefile evaluate.py
import json, os, pickle, subprocess, sys

try:
    import sacrebleu
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sacrebleu"])
    import sacrebleu

EXAMPLES = [
    {
        "rid": 0,
        "src": "Это пример текста для перевода!",
        "ref": "Ари аиҭагаразы атекст аҿырԥштəы ауп!"
    },
    {
        "rid": 1,
        "src": "Абхазский язык — один из древнейших языков мира",
        "ref": "Аԥсуа бызшәа — адунеи аҿы ижәытәӡатәиу абызшәақәа ируакуп"
    },
    {
        "rid": 2,
        "src": "Кириллица стала основой абхазской письменности в 1954 году",
        "ref": "Акириллица аԥсуа ҩыра шьаҭас иаиуит 1954 шықәсазы"
    },
    {
        "rid": 3,
        "src": "В абхазском языке насчитывается свыше 80 звуков",
        "ref": "Аԥсуа бызшәаҿы 80 бжьы иреиҳауп"
    },
    {
        "rid": 4,
        "src": "По данным на 2021 год, в Абхазии на абхазском языке говорило около 100 тысяч человек",
        "ref": "2021 шықәсазы иҟоу аинформациа ала, Аԥсны аԥсышәала ицәажәон 100 нызқьҩык ауаа раҟара"
    }
]

def main():
    inp = [{"rid": e["rid"], "src": e["src"]} for e in EXAMPLES]
    with open("input.pickle", "wb") as f:
        pickle.dump(inp, f)

    print("Running solution.py...")
    res = subprocess.run([sys.executable, "solution.py"], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"ERROR running solution.py: {res.stderr}")
        return

    with open("output.json", "r") as f:
        outputs = json.load(f)

    out_map = {o["rid"]: o["translation"] for o in outputs}

    scores = []
    print("\nRESULTS:")
    for e in EXAMPLES:
        hyp = out_map.get(e["rid"], "")
        ref = e["ref"]
        bleu = sacrebleu.sentence_bleu(hyp, [ref]).score
        scores.append(bleu)
        print(f"[{e['rid']}] BLEU: {bleu:.2f} | {hyp[:60]}...")

    avg_bleu = sum(scores) / len(scores)
    print(f"\nFINAL VALIDATION BLEU-SCORE: {avg_bleu:.2f}")

if __name__ == "__main__":
    main()


Writing evaluate.py


In [ ]:
!git pull origin main
!python evaluate.py

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 2.27 KiB | 2.27 MiB/s, done.
From https://github.com/weissv/task1
 * branch            main       -> FETCH_HEAD
   aab7d49..83d2328  main       -> origin/main
Updating aab7d49..83d2328
Fast-forward
 solution.py | 130 ++++++++++++++++++++++++++++++++++++++----------------------
 1 file changed, 82 insertions(+), 48 deletions(-)
Running solution.py (Two-Pass Pipeline)...

RESULTS:
[1] BLEU: 100.00 | Если вы хотите сделать форк проекта на GitHub... Кнопка 'For...
[2] BLEU: 100.00 | Он рассказал в интервью: 'Я сделан из металла'....
[3] BLEU: 84.24 | Даниус сказал, что она была готова, когда вошла в комнату....
[4] BLEU: 37.34 | Для создания новой ветки используйте команду 'git branch'. П...
[5] BLEU: 75.22 | Джордан объяснила, что она развернула приложение до тог